# Évaluation de LLMs sur des tâches de résumé (summarization)

## Ce que tu vas construire
- Des scripts pour calculer et comparer des métriques de résumé.
- Des rapports comparatifs (DataFrames, visualisations) sur plusieurs LLMs.
- Des métriques d'accuracy "maison" adaptées (ou pas) au résumé.
- Des résumés générés par plusieurs modèles pour comparaison.
- Des fonctions réutilisables pour charger les données, générer des résumés, calculer ROUGE.

---

## Avant de commencer : je ne vais pas te laisser avancer sans dire ce qui cloche

Tu m'as demandé d'être un mentor sans complaisance. Voici ce que je vois de problématique dans **la conception même de cet exercice**, pas seulement dans le code à écrire :

1. **L'énoncé dit d'utiliser `prompt_title` comme "référence de résumé" (reference summary) pour l'article `prompt_text`.** Si c'est vraiment ça — et je n'ai pas accès au fichier réel, donc je pars de ce que l'énoncé dit littéralement — **c'est une erreur de conception, pas un détail.** Un titre n'est pas un résumé. Un titre fait 5-10 mots, un résumé fait plusieurs phrases et doit couvrir les points clés. Comparer un résumé généré (probablement 2-3 phrases) à un titre de 8 mots avec ROUGE-N va mécaniquement produire des scores bas ou incohérents, **indépendamment de la qualité réelle du résumé généré**. Tu vas mesurer "est-ce que mon résumé ressemble à un titre", pas "est-ce que mon résumé est bon". Je le signale explicitement dans le code plus bas — ne prends pas ces scores ROUGE comme une évaluation valide de la qualité des résumés tant que tu n'as pas vérifié ce que contient réellement ta colonne de référence.
2. **La Partie IV te fait calculer une "accuracy" par correspondance exacte sur du texte généré.** L'énoncé te demande ensuite de "discuter pourquoi c'est probablement très bas ou zéro" — ce qui est une façon polie de dire que **cette métrique n'a aucun sens pour du texte généré**. Deux résumés peuvent être sémantiquement identiques sans partager une seule séquence de mots identique. Ne te contente pas de constater que le score est nul : comprends que c'est le signe que la métrique elle-même est inadaptée à la tâche, pas que les modèles sont mauvais.
3. **Comparer `t5-small`, `t5-base` et `gpt2` "à armes égales" est trompeur si tu ne le précises pas clairement dans tes conclusions.** T5 est entraîné avec un préfixe `"summarize: "` — c'est littéralement une tâche pour laquelle il a été fine-tuné (sur CNN/DailyMail entre autres). GPT-2, lui, n'a **jamais** été entraîné spécifiquement pour résumer ; le hack `"TL;DR:"` vient du papier original GPT-2 qui montrait un résumé *zero-shot* de qualité médiocre. Si GPT-2 obtient des scores ROUGE plus bas que T5, ce n'est pas une découverte surprenante ni une preuve que "les gros modèles causaux sont mauvais en résumé" — c'est attendu par construction. Ne présente pas cette comparaison comme équitable dans ton rapport final si tu ne mentionnes pas cette asymétrie.
4. **Je n'ai pas accès au dataset réel** (le lien de téléchargement n'est pas accessible depuis mon environnement). Je vais écrire du code qui suppose des colonnes `prompt_text` et `prompt_title` d'après l'énoncé, avec des vérifications défensives. **Vérifie toi-même les noms de colonnes réels avec `.columns` avant de faire confiance à mon code** — si ça ne correspond pas, adapte, ne force pas.

Ceci dit, l'exercice a un vrai mérite pédagogique : il te fait *découvrir par toi-même* que accuracy est absurde ici (Partie IV) et il te fait manipuler ROUGE dans des cas limites (Partie VI) pour que tu comprennes ses failles avant de l'utiliser aveuglément en Partie VII-VIII. Ça, c'est du bon enseignement — même si l'exécution du choix de référence (titre au lieu de vrai résumé) semble bancale.

## Partie I : Setup

In [ ]:
%pip install -q rouge_score==0.1.2
%pip install -q evaluate
%pip install -qU accelerate
%pip install -q datasets
%pip install -q nltk
%pip install -q transformers torch pandas

In [ ]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

In [ ]:
import gc
import torch
import pandas as pd
import numpy as np
import evaluate
from nltk.tokenize import sent_tokenize
from transformers import (
    T5ForConditionalGeneration,
    AutoTokenizer,
    GPT2LMHeadModel,
    GPT2Tokenizer,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé : {device}")

## Partie II : Chargement et exploration du dataset

In [ ]:
# ⚠️ Adapte ces chemins à l'endroit où tu as téléchargé les fichiers.
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Colonnes train.csv :", train_df.columns.tolist())
print("Colonnes test.csv  :", test_df.columns.tolist())

# Vérification défensive : si ces colonnes n'existent pas chez toi, arrête-toi ici
# et adapte les noms de colonnes utilisés dans tout ce notebook.
required_cols = {"prompt_text", "prompt_title"}
assert required_cols.issubset(train_df.columns), (
    f"Colonnes attendues manquantes : {required_cols - set(train_df.columns)}. "
    "Regarde `.columns` ci-dessus et adapte le code."
)

In [ ]:
SEED = 42

train_sample = train_df.sample(n=min(100, len(train_df)), random_state=SEED).reset_index(drop=True)
test_sample = test_df.sample(n=min(50, len(test_df)), random_state=SEED).reset_index(drop=True)

print(f"Échantillon train : {len(train_sample)} lignes")
print(f"Échantillon test  : {len(test_sample)} lignes")

In [ ]:
print("Article (prompt_text) :\n")
print(train_sample.loc[0, "prompt_text"][:1000], "...\n")
print("Référence utilisée comme 'résumé' (prompt_title) :\n")
print(train_sample.loc[0, "prompt_title"])

print("\n--- Regarde la différence de longueur ci-dessus. ---")
print("Si le titre fait 5 mots et l'article 500, garde bien en tête la mise en garde de l'intro :")
print("ROUGE va comparer un résumé généré multi-phrases à ce titre. Ce n'est pas anodin.")

In [ ]:
display(train_sample)
display(test_sample)

## Partie III : Résumé avec T5

In [ ]:
def batch_generator(data, batch_size):
    """Découpe une liste en lots de taille batch_size."""
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

In [ ]:
def summarize_with_t5(texts, model_name="t5-small", batch_size=8, max_input_length=512, max_output_length=64):
    """
    Génère des résumés pour une liste de textes avec un modèle T5.

    NOTE : le tokenizer et le modèle sont chargés une seule fois, PAS à chaque appel
    de fonction si tu appelles cette fonction plusieurs fois pour le même modèle —
    déplace le chargement en dehors si tu boucles sur plusieurs sous-ensembles.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
    model.eval()

    summaries = []

    for batch in batch_generator(texts, batch_size):
        # Préfixe "summarize: " requis par T5, qui est un modèle multi-tâches
        # conditionné par préfixe textuel ("translate English to French: ", etc.)
        inputs = ["summarize: " + t for t in batch]

        encoded = tokenizer(
            inputs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **encoded,
                max_length=max_output_length,
                num_beams=4,
                early_stopping=True
            )

        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        summaries.extend(decoded)

        # Nettoyage mémoire après chaque batch
        del encoded, output_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return summaries

In [ ]:
articles = train_sample["prompt_text"].tolist()
references = train_sample["prompt_title"].tolist()

t5_small_summaries = summarize_with_t5(articles, model_name="t5-small")

results_df = pd.DataFrame({
    "article": articles,
    "reference": references,
    "t5_small_summary": t5_small_summaries
})
display(results_df)

## Partie IV : Évaluation par "accuracy"

In [ ]:
def exact_match_accuracy(predictions, references):
    """Proportion de prédictions strictement identiques (après normalisation basique) aux références."""
    correct = sum(
        p.strip().lower() == r.strip().lower()
        for p, r in zip(predictions, references)
    )
    return correct / len(references)

In [ ]:
acc = exact_match_accuracy(t5_small_summaries, references)
print(f"Accuracy (correspondance exacte) : {acc:.4f}")

**Pourquoi ce score est probablement nul (ou proche de zéro), et pourquoi ce n'est PAS le signe que T5 est un mauvais modèle :**

1. T5 génère des phrases complètes ; la référence est un titre de quelques mots. Une correspondance exacte entre les deux est structurellement quasi impossible, indépendamment de la qualité sémantique du résumé.
2. Même en comparant deux vrais résumés (générés vs référence humaine), la correspondance exacte mot pour mot est un critère bien trop strict : deux paraphrases parfaitement valides d'un même contenu ne partagent souvent pas une seule séquence identique de mots.
3. L'accuracy par correspondance exacte est adaptée à des tâches à réponse fermée (classification, QA à réponse courte extraite mot pour mot) — **pas** à de la génération de texte libre. L'utiliser ici, c'est appliquer le mauvais outil à la tâche, pas une négligence du modèle.

## Partie V : Implémentation de la métrique ROUGE

In [ ]:
rouge_metric = evaluate.load("rouge")

In [ ]:
def compute_rouge_score(predictions, references):
    """
    Calcule les scores ROUGE entre des prédictions et des références.

    Le calcul ROUGE-Lsum (résumé multi-phrases) attend un format où chaque phrase
    est séparée par un retour à la ligne. On utilise le tokenizer de phrases nltk
    (`sent_tokenize`) plutôt qu'un simple split sur les points, parce qu'un split
    naïf casse sur les abréviations ("M. Dupont", "etc.", "U.S.") et fausse le découpage.
    """
    def add_newlines(text):
        return "\n".join(sent_tokenize(text.strip())) if text.strip() else ""

    predictions_formatted = [add_newlines(p) for p in predictions]
    references_formatted = [add_newlines(r) for r in references]

    scores = rouge_metric.compute(
        predictions=predictions_formatted,
        references=references_formatted,
        use_stemmer=True
    )
    return scores

In [ ]:
rouge_scores_t5_small = compute_rouge_score(t5_small_summaries, references)
print(rouge_scores_t5_small)

**Rappel, encore une fois** : ces scores comparent un résumé multi-phrases à un titre. Ne les interprète pas comme "la qualité de résumé de t5-small en absolu" tant que tu n'as pas vérifié si `prompt_title` est vraiment ce que tu penses qu'il est.

## Partie VI : Comprendre les scores ROUGE (cas limites)

In [ ]:
# 1. Correspondance exacte : les résumés générés sont identiques aux références
identical_preds = references.copy()
print("Cas 1 - Correspondance exacte :")
print(compute_rouge_score(identical_preds, references))
print("Attendu : tous les scores (rouge1, rouge2, rougeL, rougeLsum) proches de 1.0\n")

In [ ]:
# 2. Prédictions vides
empty_preds = ["" for _ in references]
print("Cas 2 - Prédictions vides :")
print(compute_rouge_score(empty_preds, references))
print("Attendu : tous les scores à 0.0 (aucun overlap possible avec une chaîne vide)\n")

In [ ]:
# 3. Effet du stemming
pred_stem = "The cats are running quickly in the gardens."
ref_stem = "The cat is running quickly in the garden."

print("Sans stemming (use_stemmer=False) :")
print(rouge_metric.compute(predictions=[pred_stem], references=[ref_stem], use_stemmer=False))

print("\nAvec stemming (use_stemmer=True) :")
print(rouge_metric.compute(predictions=[pred_stem], references=[ref_stem], use_stemmer=True))

print("\nAttendu : le score augmente avec le stemming, car 'cats'/'cat' et 'gardens'/'garden'")
print("deviennent identiques après réduction à leur racine. Le stemming récompense la similarité")
print("morphologique, pas nécessairement la similarité de sens.")

In [ ]:
# 4. Effet du degré de chevauchement de n-grammes sur ROUGE-1 vs ROUGE-2
reference_text = "The quick brown fox jumps over the lazy dog"

variants = {
    "identique": "The quick brown fox jumps over the lazy dog",
    "ordre_different": "The dog lazy over jumps fox brown quick the",
    "mots_partages_partiels": "A quick brown fox leaps over a sleepy dog",
    "aucun_mot_commun": "Bright stars shine above the silent mountain",
}

for label, text in variants.items():
    scores = rouge_metric.compute(predictions=[text], references=[reference_text], use_stemmer=True)
    print(f"{label:25s} rouge1={scores['rouge1']:.3f}  rouge2={scores['rouge2']:.3f}")

print("\nCe qu'il faut observer : 'ordre_different' contient exactement les mêmes mots que la")
print("référence, dans un ordre absurde en anglais. ROUGE-1 (unigrammes) reste très élevé car il")
print("ignore l'ordre. ROUGE-2 (bigrammes) chute fortement, car l'ordre des mots adjacents compte.")
print("Ça illustre une vraie limite de ROUGE-1 : un texte peut être un charabia complet et quand")
print("même obtenir un bon score ROUGE-1 s'il recycle le bon vocabulaire.")

In [ ]:
# 5. Symétrie de ROUGE par rapport à predictions/references
text_a = "The cat sat on the mat"
text_b = "A cat was sitting on a mat in the house"

scores_ab = rouge_metric.compute(predictions=[text_a], references=[text_b], use_stemmer=True)
scores_ba = rouge_metric.compute(predictions=[text_b], references=[text_a], use_stemmer=True)

print("predictions=A, references=B :", scores_ab)
print("predictions=B, references=A :", scores_ba)
print("\nrouge1/rouge2 (F-mesure) sont symétriques par construction (moyenne harmonique de")
print("precision et recall, qui s'échangent quand on inverse predictions/references).")
print("Vérifie si c'est bien ce que tu observes ci-dessus — si les deux lignes divergent,")
print("regarde la variante rougeLsum, qui peut différer légèrement selon l'implémentation.")

## Partie VII : Comparer petits et grands modèles

In [ ]:
def summarize_with_gpt2(texts, model_name="gpt2", batch_size=4, max_new_tokens=64, max_input_length=900):
    """
    Génère des "résumés" avec GPT-2 via le prompt 'TL;DR:' (technique popularisée par le papier
    GPT-2 original pour du résumé zero-shot). GPT-2 n'a pas de token de padding par défaut,
    et sa fenêtre de contexte est limitée à 1024 tokens (texte d'entrée + texte généré compris),
    d'où le besoin explicite de tronquer l'article en amont.
    """
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token  # GPT-2 n'a pas de pad_token par défaut
    model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
    model.eval()

    summaries = []

    for batch in batch_generator(texts, batch_size):
        prompts = [t + "\nTL;DR:" for t in batch]

        encoded = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length  # laisse de la place pour max_new_tokens sous la limite de 1024
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        for i, ids in enumerate(output_ids):
            full_text = tokenizer.decode(ids, skip_special_tokens=True)
            # On ne garde que ce qui vient APRÈS "TL;DR:", sinon on réévalue l'article lui-même
            if "TL;DR:" in full_text:
                summary = full_text.split("TL;DR:")[-1].strip()
            else:
                summary = full_text.strip()
            summaries.append(summary)

        del encoded, output_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return summaries

In [ ]:
t5_base_summaries = summarize_with_t5(articles, model_name="t5-base")
gpt2_summaries = summarize_with_gpt2(articles, model_name="gpt2")

In [ ]:
rouge_t5_small = compute_rouge_score(t5_small_summaries, references)
rouge_t5_base = compute_rouge_score(t5_base_summaries, references)
rouge_gpt2 = compute_rouge_score(gpt2_summaries, references)

print("t5-small :", rouge_t5_small)
print("t5-base  :", rouge_t5_base)
print("gpt2     :", rouge_gpt2)

In [ ]:
def compute_rouge_per_row(predictions, references):
    """Calcule un score ROUGE indépendant pour chaque paire (prédiction, référence),
    plutôt qu'une seule moyenne globale. Utile pour repérer les cas particulièrement
    mauvais qui seraient noyés dans une moyenne agrégée."""
    rows = []
    for pred, ref in zip(predictions, references):
        score = compute_rouge_score([pred], [ref])
        rows.append(score)
    return pd.DataFrame(rows)

In [ ]:
per_row_t5_small = compute_rouge_per_row(t5_small_summaries, references)
per_row_t5_base = compute_rouge_per_row(t5_base_summaries, references)
per_row_gpt2 = compute_rouge_per_row(gpt2_summaries, references)

print("Distribution des scores ROUGE-1 par ligne, t5-small :")
display(per_row_t5_small.describe())

print("\nDistribution des scores ROUGE-1 par ligne, gpt2 :")
display(per_row_gpt2.describe())

**Ce que la moyenne globale te cache et que le per-row révèle** : une moyenne peut être décente alors que la moitié des lignes sont catastrophiques et l'autre moitié excellentes. Regarde l'écart-type et les valeurs min/max dans `.describe()`, pas seulement la moyenne — sinon tu tires des conclusions sur "la performance du modèle" alors que tu n'as observé qu'un résumé statistique grossier de sa performance.

## Partie VIII : Comparer tous les modèles

In [ ]:
def compare_models(model_scores: dict):
    """
    model_scores : dict {nom_du_modele: dict_scores_rouge_moyens}
    Retourne un DataFrame comparatif, une ligne par modèle.
    """
    return pd.DataFrame(model_scores).T

In [ ]:
comparison_table = compare_models({
    "t5-small": rouge_t5_small,
    "t5-base": rouge_t5_base,
    "gpt2": rouge_gpt2,
})
display(comparison_table)

In [ ]:
def compare_models_summaries(articles, references, **model_summaries):
    """
    model_summaries : kwargs, ex. t5_small=t5_small_summaries, gpt2=gpt2_summaries
    Retourne un DataFrame avec une colonne par modèle, pour comparaison côte à côte.
    """
    data = {"article": articles, "reference": references}
    data.update(model_summaries)
    return pd.DataFrame(data)

In [ ]:
side_by_side = compare_models_summaries(
    articles,
    references,
    t5_small=t5_small_summaries,
    t5_base=t5_base_summaries,
    gpt2=gpt2_summaries,
)
display(side_by_side)

## Bilan sans complaisance

- **Ne publie pas `comparison_table` comme si c'était une évaluation objective et équitable.** Tu compares un modèle fine-tuné pour le résumé (T5) à un modèle qui ne l'est pas du tout (GPT-2, avec un hack de prompt). Si ton rapport final présente juste un tableau de scores sans cette mise en garde, il induit en erreur quiconque le lit sans avoir suivi tout le raisonnement.
- **Le choix de `prompt_title` comme référence reste, à mes yeux, le vrai problème méthodologique de cet exercice**, pas un détail que je signale par excès de prudence. Si tu peux accéder au dataset réel et qu'il existe une vraie colonne de résumé de référence (souvent nommée `summary`, `highlights`, ou similaire selon le dataset d'origine), utilise-la à la place et refais tourner Parties III à VIII. Sinon, tout ce que produit ce notebook mesure "la ressemblance à un titre", pas "la qualité de résumé" — et il faut le dire explicitement dans tes conclusions, pas le glisser sous le tapis parce que ROUGE donne quand même un nombre.
- ROUGE lui-même reste une métrique de surface (chevauchement de n-grammes), pas de sens. Un résumé faux mais qui recycle le bon vocabulaire peut scorer haut ; un résumé correct mais reformulé peut scorer bas. Ne présente jamais un score ROUGE seul comme preuve de qualité — accompagne-le d'une lecture qualitative d'un échantillon de résumés (ce que `compare_models_summaries` te permet de faire), ou d'une métrique complémentaire comme BERTScore si tu veux capturer la similarité sémantique.